# CBLV Node Classification with GNN

This notebook builds DGL graphs from phylogenetic trees for node-level prediction tasks.

## Pipeline Overview
1. **Load Trees** - Parse BEAST2 tree files and analyze structure
2. **CBLV Encoding** - Extract topology features per location via virtual subtree traversal
3. **DTW Features** - Compute epidemic curve similarity between location pairs
4. **Graph Construction** - Build DGL graphs with node/edge features and labels
5. **Validation** - Verify features and labels against source files

## Feature Summary
| Component | Description |
|-----------|-------------|
| Node features | CBLV encoding (tree_width × 4), rescaled to [0,1] |
| Edge features | DTW distance + lag statistics (3 dims), Z-normalized |
| Node labels | R0 (reproduction number), Source_Sink_Score |

In [1]:
# Cell 1: Configuration and Tree Analysis
"""
Load BEAST2 tree files and compute summary statistics.
Determines tree_width (max tips per location) for CBLV padding.
"""
import os, sys, re
import numpy as np
from pathlib import Path
from collections import defaultdict
import dendropy as dp

sys.path.insert(0, str(Path(os.getcwd()).parent / 'utils'))

# === CONFIGURATION ===
INPUT_FOLDER = Path('/Users/lukelyu/Desktop/epidata/500_1_MM0.002')
FILE_PATTERN = '*_beast2.trees'


def load_tree(tree_file, tree_idx=0):
    """Load and preprocess a single tree from a BEAST2 file."""
    tree_list = dp.TreeList.get(
        path=str(tree_file), schema='nexus',
        suppress_internal_node_taxa=True, suppress_leaf_node_taxa=True
    )
    phy = tree_list[tree_idx]
    phy.is_rooted = True
    phy.suppress_unifurcations()  # Collapse single-child nodes
    phy.calc_node_root_distances()
    return phy


def get_location(node):
    """Extract location ID from node annotation (e.g., 'I{7}' -> 7)."""
    annot = node.annotations.get_value('type') if node.annotations else None
    return int(str(annot).split('{')[1].split('}')[0]) if annot and '{' in str(annot) else None


def is_sample(node):
    """Check if node is an actual sample (not ancestral reconstruction)."""
    return node.annotations and node.annotations.get_value('samp') == 'sample'


def analyze_trees(input_folder, file_pattern):
    """Analyze all trees and return summary info + max tips per location."""
    tree_files = sorted(input_folder.glob(file_pattern))
    all_info, max_tips, max_info = [], 0, None
    
    for tree_file in tree_files:
        tree_list = dp.TreeList.get(
            path=str(tree_file), schema='nexus',
            suppress_internal_node_taxa=True, suppress_leaf_node_taxa=True
        )
        for idx, phy in enumerate(tree_list):
            phy.is_rooted = True
            phy.suppress_unifurcations()
            phy.calc_node_root_distances()
            
            # Count tips per location
            pop_counts = defaultdict(int)
            for nd in phy.leaf_node_iter():
                loc = get_location(nd)
                if is_sample(nd) and loc is not None:
                    pop_counts[loc] += 1
            
            # Track maximum
            for loc, count in pop_counts.items():
                if count > max_tips:
                    max_tips, max_info = count, {'file': tree_file.name, 'idx': idx, 'loc': loc}
            
            all_info.append({
                'file': tree_file.name, 'idx': idx,
                'n_tips': len(phy.leaf_nodes()),
                'height': max(nd.root_distance for nd in phy.leaf_node_iter()),
                'pop_counts': dict(pop_counts)
            })
    
    return all_info, max_tips, max_info, tree_files


# Run analysis
all_trees_info, max_tips_per_location, max_tips_info, tree_files = analyze_trees(INPUT_FOLDER, FILE_PATTERN)

print(f"{'='*60}\nTREE SUMMARY\n{'='*60}")
print(f"Input: {INPUT_FOLDER} ({len(tree_files)} files, {len(all_trees_info)} trees)")
print(f"Locations: {len(all_trees_info[0]['pop_counts'])}")
print(f"Tips: {min(t['n_tips'] for t in all_trees_info)}-{max(t['n_tips'] for t in all_trees_info)}")
print(f"Height: {min(t['height'] for t in all_trees_info):.1f}-{max(t['height'] for t in all_trees_info):.1f}")
print(f"\nMax tips/location: {max_tips_per_location} → tree_width = {max_tips_per_location}")

TREE SUMMARY
Input: /Users/lukelyu/Desktop/epidata/500_1_MM0.002 (500 files, 500 trees)
Locations: 16
Tips: 1383-5792
Height: 242.8-1191.2

Max tips/location: 531 → tree_width = 531


In [ ]:
# Cell 2: CBLV Node Feature Encoder
"""
Compact Branch-Length Vector (CBLV) encoding via virtual subtree traversal.
Encodes tree topology as a fixed-size matrix per location without copying trees.

CBLV Matrix Structure (tree_width × 4):
  - Col 0: Leaf distances from previous branching point
  - Col 1: Internal node depths (root distances)
  - Col 2: Accumulated edge lengths to leaves
  - Col 3: Accumulated edge lengths to internal nodes
"""

class VirtualSubtreeEncoder:
    """CBLV encoder using virtual subtree traversal (no tree copying)."""
    
    def __init__(self, phy, tree_height):
        self.phy = phy
        self.tree_height = tree_height
        self._precompute_location_stats()

    def _precompute_location_stats(self):
        """Bottom-up pass: compute location counts and max distances per node."""
        for nd in self.phy.postorder_node_iter():
            if nd.is_leaf():
                loc = get_location(nd)
                if is_sample(nd) and loc is not None:
                    nd.loc_counts = {loc: 1}
                    nd.loc_max_dist = {loc: nd.root_distance}
                else:
                    nd.loc_counts, nd.loc_max_dist = {}, {}
            else:
                nd.loc_counts, nd.loc_max_dist = {}, {}
                for child in nd.child_nodes():
                    for loc, cnt in getattr(child, 'loc_counts', {}).items():
                        nd.loc_counts[loc] = nd.loc_counts.get(loc, 0) + cnt
                    for loc, dist in getattr(child, 'loc_max_dist', {}).items():
                        nd.loc_max_dist[loc] = max(nd.loc_max_dist.get(loc, 0), dist)

    def _find_mrca(self, loc):
        """Find MRCA of all tips with target location."""
        total = self.phy.seed_node.loc_counts.get(loc, 0)
        if total < 2:
            return None
        mrca = self.phy.seed_node
        while True:
            children = [c for c in mrca.child_nodes() if c.loc_counts.get(loc, 0) == total]
            if len(children) == 1:
                mrca = children[0]
            else:
                break
        return mrca

    def _is_branch_point(self, node, loc):
        """Check if node is a branching point for target location."""
        return not node.is_leaf() and sum(1 for c in node.child_nodes() if c.loc_counts.get(loc, 0) > 0) >= 2

    def _find_parent_branch(self, node, loc, mrca):
        """Find nearest ancestor that is a branching point for target location."""
        current = node.parent_node
        while current and current != mrca:
            if self._is_branch_point(current, loc):
                return current
            current = current.parent_node
        return mrca

    def _accum_edge(self, node, parent_branch):
        """Compute accumulated edge length from node up to parent branching point."""
        total, current = 0, node
        while current and current != parent_branch:
            total += current.edge.length or 0
            current = current.parent_node
        return total

    def _virtual_inorder(self, node, loc, last_branch_dist, mrca):
        """Generator for virtual in-order traversal of location subtree."""
        if node.is_leaf():
            if is_sample(node) and get_location(node) == loc:
                parent = self._find_parent_branch(node, loc, mrca)
                yield ('leaf', node.root_distance - last_branch_dist, self._accum_edge(node, parent))
        elif self._is_branch_point(node, loc):
            # Sort children by max distance (deepest first)
            children = sorted(
                [c for c in node.child_nodes() if c.loc_counts.get(loc, 0) > 0],
                key=lambda c: c.loc_max_dist.get(loc, 0), reverse=True
            )
            yield from self._virtual_inorder(children[0], loc, last_branch_dist, mrca)
            
            accum = (node.edge.length or 0) if node == mrca else self._accum_edge(node, self._find_parent_branch(node, loc, mrca))
            yield ('internal', node.root_distance, accum)
            
            for child in children[1:]:
                yield from self._virtual_inorder(child, loc, node.root_distance, mrca)
        else:
            relevant = [c for c in node.child_nodes() if c.loc_counts.get(loc, 0) > 0]
            if relevant:
                yield from self._virtual_inorder(relevant[0], loc, last_branch_dist, mrca)

    def encode_cblv(self, loc, tree_width=None, rescale=True):
        """
        Encode CBLV for a target location.
        
        Returns:
            heights: (tree_width, 4) matrix of CBLV features
            stem: Distance from root to MRCA
            n_tips: Number of tips at this location
        """
        mrca = self._find_mrca(loc)
        if mrca is None:
            return np.zeros((tree_width or 1, 4)), 0, self.phy.seed_node.loc_counts.get(loc, 0)

        stem = mrca.root_distance
        n_tips = mrca.loc_counts.get(loc, 0)
        heights = np.zeros((n_tips, 4))
        idx = 0

        for event_type, val1, val2 in self._virtual_inorder(mrca, loc, mrca.root_distance, mrca):
            if idx >= n_tips:
                break
            if event_type == 'leaf':
                heights[idx, 0] = val1 + (stem if idx == 0 else 0)
                heights[idx, 2] = val2
            else:  # internal
                if idx + 1 < n_tips:
                    heights[idx + 1, 1] = val1
                    heights[idx + 1, 3] = val2
                idx += 1

        if rescale:
            heights /= self.tree_height

        # Pad to tree_width
        if tree_width and n_tips != tree_width:
            padded = np.zeros((tree_width, 4))
            padded[:min(n_tips, tree_width)] = heights[:min(n_tips, tree_width)]
            heights = padded

        return heights, stem, n_tips

    def get_all_locations(self):
        """Get sorted list of all sample locations in the tree."""
        return sorted({get_location(nd) for nd in self.phy.leaf_node_iter() 
                      if is_sample(nd) and get_location(nd) is not None})

print("CBLV encoder ready")

In [ ]:
# Cell 3: DTW Edge Feature Extraction
"""
Dynamic Time Warping (DTW) for epidemic curve similarity.
Computes pairwise DTW distances and lag statistics between location-specific
tip time distributions (smoothed via KDE).

Edge Features (3 dims):
  - DTW distance: Curve dissimilarity measure
  - Mean lag: Average temporal offset in alignment
  - Lag std: Variability in temporal alignment
"""
from scipy.stats import gaussian_kde


def extract_tip_times(tree_file, tree_idx=0):
    """Extract tip sampling times grouped by location from BEAST2 tree file."""
    with open(tree_file) as f:
        trees = re.findall(r'tree STATE_\d+ = (.+?)(?=\ntree |\nEnd;|$)', f.read(), re.DOTALL)
    if not trees or tree_idx >= len(trees):
        return {}
    
    tips = defaultdict(list)
    for loc, time in re.findall(r'\d+\[&type="I\{(\d+)\}",samp="sample",time=([\d.]+)\]', trees[tree_idx]):
        tips[int(loc)].append(float(time))
    return dict(tips)


def tips_to_curves(tips_by_loc, num_points=200):
    """Convert tip times to KDE-smoothed epidemic curves."""
    all_times = [t for times in tips_by_loc.values() for t in times]
    if not all_times:
        return {}, 0
    
    t_grid = np.linspace(min(all_times), max(all_times), num_points)
    dt = (max(all_times) - min(all_times)) / (num_points - 1)
    
    curves = {}
    for loc, times in tips_by_loc.items():
        curves[loc] = gaussian_kde(times)(t_grid) * len(times) if len(times) >= 2 else np.zeros(num_points)
    return curves, dt


def dtw_with_lag(curve_a, curve_b):
    """Compute DTW distance and lag statistics between two curves."""
    n, m = len(curve_a), len(curve_b)
    cost = (curve_a[:, None] - curve_b[None, :]) ** 2
    
    # Forward pass
    dtw = np.full((n + 1, m + 1), np.inf)
    dtw[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dtw[i, j] = cost[i-1, j-1] + min(dtw[i-1, j-1], dtw[i-1, j], dtw[i, j-1])
    
    # Backtrack for lag computation
    lags, i, j = [], n, m
    while i > 0 and j > 0:
        lags.append(j - i)
        step = np.argmin([dtw[i-1, j-1], dtw[i-1, j], dtw[i, j-1]])
        i, j = (i-1, j-1) if step == 0 else ((i-1, j) if step == 1 else (i, j-1))
    
    lags = np.array(lags)
    return dtw[n, m], lags.mean(), lags.std()


def compute_dtw_edge_features(tree_file, tree_idx=0):
    """Compute DTW-based edge features for all location pairs."""
    tips_by_loc = extract_tip_times(tree_file, tree_idx)
    if not tips_by_loc:
        return None, None, None, []
    
    curves, dt = tips_to_curves(tips_by_loc)
    if not curves:
        return None, None, None, []
    
    locations = sorted(curves.keys())
    curve_arr = np.array([curves[loc] for loc in locations])
    
    src, dst, feats = [], [], []
    for i in range(len(locations)):
        for j in range(len(locations)):
            if i != j:
                dist, lag_mean, lag_std = dtw_with_lag(curve_arr[i], curve_arr[j])
                src.append(i)
                dst.append(j)
                feats.append([dist, lag_mean * dt, lag_std * dt])
    
    return src, dst, np.array(feats), locations

print("DTW edge features ready")

In [ ]:
# Cell 4: DGL Graph Builder
"""
Build DGL graphs with CBLV node features, DTW edge features, and ground-truth labels.

Graph Structure:
  - Nodes: Locations (fully connected)
  - Node features: Flattened CBLV matrix (tree_width × 4)
  - Edge features: DTW distance + lag stats (3 dims)
  - Labels: R0, Source_Sink_Score per node
"""
import dgl
import torch
from tqdm import tqdm
from trajectory_utils import load_trajectory_wide, load_reactions_from_xml, load_R0_and_population_from_csv
from node_feature import classify_events_for_sample, calculate_node_event_metrics


def extract_labels(input_folder, file_prefix, tree_idx, num_nodes):
    """Extract R0 and Source_Sink_Score labels for each node."""
    input_folder = Path(input_folder)
    
    # R0 from parameter CSV
    params = load_R0_and_population_from_csv(str(input_folder / f"{file_prefix}_parameter.csv"))
    r0 = np.array([params['R0'].get(i, 0.0) for i in range(num_nodes)], dtype=np.float32)
    
    # Source_Sink_Score from trajectory analysis
    df = load_trajectory_wide(str(input_folder / f"{file_prefix}_beast2.traj"))
    reactions = load_reactions_from_xml(str(input_folder / f"{file_prefix}_beast2.xml"))
    species_cols = [c for c in df.columns if c not in ['Sample', 't']]
    
    sample_data = df[df['Sample'] == tree_idx].sort_values('t').reset_index(drop=True)
    events = classify_events_for_sample(sample_data, reactions, species_cols)
    event_counts = dict(events['event_type'].value_counts()) if not events.empty else {}
    metrics = calculate_node_event_metrics(event_counts, num_nodes)
    sss = np.array([metrics[i]['source_sink_score'] for i in range(num_nodes)], dtype=np.float32)
    
    return {'R0': r0, 'Source_Sink_Score': sss}


def build_graph(tree_file, tree_idx, tree_width, input_folder):
    """Build a single DGL graph from a tree."""
    phy = load_tree(tree_file, tree_idx)
    tree_height = max(nd.root_distance for nd in phy.leaf_node_iter())
    
    # Node features (CBLV)
    encoder = VirtualSubtreeEncoder(phy, tree_height)
    locations = encoder.get_all_locations()
    n_nodes = len(locations)
    
    node_feats = np.zeros((n_nodes, tree_width * 4))
    for i, loc in enumerate(locations):
        cblv, _, _ = encoder.encode_cblv(loc, tree_width=tree_width, rescale=True)
        node_feats[i] = cblv.flatten()
    
    # Edge features (DTW)
    src, dst, edge_feats, dtw_locs = compute_dtw_edge_features(tree_file, tree_idx)
    assert locations == dtw_locs, f"Location mismatch: CBLV={locations}, DTW={dtw_locs}"
    
    # Labels
    file_prefix = Path(tree_file).stem.replace('_beast2', '')
    labels = extract_labels(input_folder, file_prefix, tree_idx, n_nodes)
    
    # Construct graph
    g = dgl.graph((src, dst), num_nodes=n_nodes)
    g.ndata['feat'] = torch.tensor(node_feats, dtype=torch.float32)
    g.ndata['location'] = torch.tensor(locations, dtype=torch.long)
    g.ndata['R0'] = torch.tensor(labels['R0'])
    g.ndata['Source_Sink_Score'] = torch.tensor(labels['Source_Sink_Score'])
    g.edata['feat'] = torch.tensor(edge_feats, dtype=torch.float32)
    
    return g, locations, tree_height


def build_all_graphs(input_folder, tree_width, file_pattern='*_beast2.trees'):
    """Build DGL graphs from all trees in folder."""
    input_folder = Path(input_folder)
    graphs = []
    
    for tree_file in tqdm(sorted(input_folder.glob(file_pattern)), desc="Building graphs"):
        file_prefix = tree_file.stem.replace('_beast2', '')
        with open(tree_file) as f:
            n_trees = len(re.findall(r'tree STATE_\d+', f.read()))
        
        for idx in range(n_trees):
            try:
                g, locs, height = build_graph(str(tree_file), idx, tree_width, input_folder)
                graphs.append((g, f"{file_prefix}_{idx}", locs, height))
            except Exception as e:
                print(f"Error {tree_file.name}[{idx}]: {e}")
    
    return graphs


def normalize_edge_features(graphs):
    """Z-score normalize edge features across all graphs."""
    all_feats = torch.cat([g.edata['feat'] for g, *_ in graphs])
    mean, std = all_feats.mean(0), all_feats.std(0)
    
    for g, *_ in graphs:
        g.edata['feat'] = (g.edata['feat'] - mean) / (std + 1e-8)
    
    return graphs

print("Graph builder ready")

In [ ]:
# Cell 5: Build and Summarize Graphs

tree_width = max_tips_per_location
graphs = normalize_edge_features(build_all_graphs(INPUT_FOLDER, tree_width))

# Summary statistics
g0 = graphs[0][0]
all_edge = torch.cat([g.edata['feat'] for g, *_ in graphs])
all_r0 = torch.cat([g.ndata['R0'] for g, *_ in graphs])
all_sss = torch.cat([g.ndata['Source_Sink_Score'] for g, *_ in graphs])

print(f"\n{'='*60}\nGRAPH SUMMARY\n{'='*60}")
print(f"Graphs: {len(graphs)} | Nodes: {g0.num_nodes()} | Edges: {g0.num_edges()}")
print(f"\nNode features (CBLV): {g0.ndata['feat'].shape} = {tree_width}×4 flattened")
print(f"  Range: [{g0.ndata['feat'].min():.4f}, {g0.ndata['feat'].max():.4f}]")
print(f"  Non-zero: {(g0.ndata['feat'] != 0).float().mean():.1%}")
print(f"\nEdge features (DTW, Z-normalized): {g0.edata['feat'].shape}")
for i, name in enumerate(['distance', 'lag_mean', 'lag_std']):
    print(f"  {name}: [{all_edge[:,i].min():.2f}, {all_edge[:,i].max():.2f}]")
print(f"\nLabels:")
print(f"  R0: [{all_r0.min():.4f}, {all_r0.max():.4f}]")
print(f"  Source_Sink_Score: [{all_sss[~all_sss.isnan()].min():.4f}, {all_sss[~all_sss.isnan()].max():.4f}]")

In [ ]:
# Cell 6: Inspect Sample Graph

g, graph_id, locs, height = graphs[0]

print(f"{'='*60}\nSAMPLE GRAPH: {graph_id} (height={height:.2f})\n{'='*60}")
print(f"Locations: {locs}\n")
print(f"{'Loc':<5} {'R0':<8} {'SSS':<10} {'CBLV[0:4]'}")
print("-" * 50)
for i in range(g.num_nodes()):
    cblv = g.ndata['feat'][i, :4].tolist()
    sss = g.ndata['Source_Sink_Score'][i].item()
    print(f"{locs[i]:<5} {g.ndata['R0'][i]:<8.4f} {sss:<10.4f} [{', '.join(f'{v:.3f}' for v in cblv)}]")

In [ ]:
# Cell 7: Validation
"""
Verify graph features and labels match source files.
Re-computes everything from scratch and compares with stored values.
"""

def validate_graph(g, graph_id, input_folder, tree_width):
    """Validate a single graph against source files."""
    file_prefix, tree_idx = graph_id.rsplit('_', 1)
    tree_idx = int(tree_idx)
    n = g.num_nodes()
    
    # Recompute labels
    labels = extract_labels(input_folder, file_prefix, tree_idx, n)
    r0_ok = np.allclose(labels['R0'], g.ndata['R0'].numpy(), rtol=1e-4)
    
    sss_exp, sss_act = labels['Source_Sink_Score'], g.ndata['Source_Sink_Score'].numpy()
    sss_ok = all(
        (np.isnan(e) and np.isnan(a)) or abs(e - a) < 1e-4
        for e, a in zip(sss_exp, sss_act)
    )
    
    # Recompute CBLV
    phy = load_tree(str(input_folder / f"{file_prefix}_beast2.trees"), tree_idx)
    tree_height = max(nd.root_distance for nd in phy.leaf_node_iter())
    encoder = VirtualSubtreeEncoder(phy, tree_height)
    
    cblv_ok = True
    for i, loc in enumerate(encoder.get_all_locations()):
        expected, _, _ = encoder.encode_cblv(loc, tree_width=tree_width, rescale=True)
        actual = g.ndata['feat'][i].numpy().reshape(tree_width, 4)
        if not np.allclose(expected, actual, rtol=1e-4):
            cblv_ok = False
            break
    
    # Edge structure check
    edge_ok = g.num_edges() == n * (n - 1) and g.edata['feat'].shape[1] == 3
    
    return {'r0': r0_ok, 'sss': sss_ok, 'cblv': cblv_ok, 'edge': edge_ok}


# Run validation on subset
print(f"{'='*60}\nVALIDATION\n{'='*60}")
n_validate = min(5, len(graphs))
results = []

for i in range(n_validate):
    g, graph_id, _, _ = graphs[i]
    r = validate_graph(g, graph_id, INPUT_FOLDER, tree_width)
    results.append(r)
    
    status = lambda x: '✓' if x else '✗'
    all_ok = all(r.values())
    print(f"{graph_id}: R0={status(r['r0'])} SSS={status(r['sss'])} CBLV={status(r['cblv'])} Edge={status(r['edge'])} → {'✓' if all_ok else '✗'}")

print(f"\n{'='*60}\nSUMMARY: {sum(all(r.values()) for r in results)}/{n_validate} passed\n{'='*60}")